# Prepare Environment

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!wget https://raw.githubusercontent.com/hackdeploy/relational-transformer/refs/heads/dev/scripts/setup_colab.py
!python setup_colab.py

--2026-02-17 02:08:12--  https://raw.githubusercontent.com/hackdeploy/relational-transformer/refs/heads/dev/scripts/setup_colab.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2728 (2.7K) [text/plain]
Saving to: ‘setup_colab.py’

setup_colab.py      100%[===================>]   2.66K  --.-KB/s    in 0s      

2026-02-17 02:08:12 (36.4 MB/s) - ‘setup_colab.py’ saved [2728/2728]

Starting Setup...
Installing Rust...
Running: curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
info: downloading installer
info: profile set to 'default'
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for 'stable-x86_64-unknown-linux-gnu'
info: latest update on 2026-02-12, rust version 1.93.1 (01f6ddf75 2026-02-11)
info: d

In [ ]:
import os
# Point HOME to your Drive folder where 'scratch/' is located
os.environ['HOME'] = "/content/drive/MyDrive/Colab_Data"

# Sample Batch Data

In [ ]:
import json
import os

def load_json_data(file_path):
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        print(f"Successfully loaded JSON from: {file_path}")
        return data
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{file_path}'. Check if it's a valid JSON file.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


In [ ]:
import json
import os

# Define the path to your JSON file within the HOME directory
json_file_path = os.path.join(os.environ['HOME'], 'relbench_datasets/rel-f1-sample-batch/sample_batch.json')

try:
    with open(json_file_path, 'r') as f:
        data = json.load(f)
        print(data['node_idxs'][0:10])
    print(f"Successfully loaded JSON from: {json_file_path}")
except FileNotFoundError:
    print(f"Error: The file '{json_file_path}' was not found.")
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from '{json_file_path}'. Check if it's a valid JSON file.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

[98308, 98308, 25649, 25649, 25649, 25649, 25649, 25649, 53685, 53685]
Successfully loaded JSON from: /content/drive/MyDrive/Colab_Data/relbench_datasets/rel-f1-sample-batch/sample_batch.json


In [ ]:
for key, value in data.items():

  if isinstance(value, list):
    val = len(value)
  else:
    val = 0

  print(f"- Key: '{key}', Value Type: {type(value)}, length: {val}")

- Key: 'node_idxs', Value Type: <class 'list'>, length: 4096
- Key: 'f2p_nbr_idxs', Value Type: <class 'list'>, length: 20480
- Key: 'table_name_idxs', Value Type: <class 'list'>, length: 4096
- Key: 'col_name_idxs', Value Type: <class 'list'>, length: 4096
- Key: 'class_value_idxs', Value Type: <class 'list'>, length: 4096
- Key: 'col_name_values', Value Type: <class 'list'>, length: 1572864
- Key: 'sem_types', Value Type: <class 'list'>, length: 4096
- Key: 'number_values', Value Type: <class 'list'>, length: 4096
- Key: 'text_values', Value Type: <class 'list'>, length: 1572864
- Key: 'datetime_values', Value Type: <class 'list'>, length: 4096
- Key: 'boolean_values', Value Type: <class 'list'>, length: 4096
- Key: 'masks', Value Type: <class 'list'>, length: 4096
- Key: 'is_targets', Value Type: <class 'list'>, length: 4096
- Key: 'is_task_nodes', Value Type: <class 'list'>, length: 4096
- Key: 'is_padding', Value Type: <class 'list'>, length: 4096
- Key: 'true_batch_size', Value T

In [ ]:
batch = load_json_data(os.path.join(os.environ['HOME'], 'relbench_datasets/rel-f1-sample-batch/sample_batch.json'))
j_nodes = load_json_data(os.path.join(os.environ['HOME'], 'scratch/pre/rel-f1/out_nodes.json'))
j_text_map = load_json_data(os.path.join(os.environ['HOME'], 'scratch/pre/rel-f1/text_map.json'))
j_column_index = load_json_data(os.path.join(os.environ['HOME'], 'scratch/pre/rel-f1/column_index.json'))

# Create reverse mappings
idx_to_text = {int(idx): text for text, idx in j_text_map.items()}
idx_to_column = {int(idx): col for col, idx in j_column_index.items()}

Successfully loaded JSON from: /content/drive/MyDrive/Colab_Data/relbench_datasets/rel-f1-sample-batch/sample_batch.json
Successfully loaded JSON from: /content/drive/MyDrive/Colab_Data/scratch/pre/rel-f1/out_nodes.json
Successfully loaded JSON from: /content/drive/MyDrive/Colab_Data/scratch/pre/rel-f1/text_map.json
Successfully loaded JSON from: /content/drive/MyDrive/Colab_Data/scratch/pre/rel-f1/column_index.json


In [ ]:
node_idx = 6
position = next(i for i, n in enumerate(j_nodes) if n['node_idx'] == node_idx)
node = j_nodes[position]

# 3. Decode the values
table_name = idx_to_text[node['table_name_idx']]
print(f"Table: {table_name}")

for i, col_idx in enumerate(node['col_name_idxs']):
    col_name = idx_to_text[col_idx]
    sem_type = node['sem_types'][i]

    if sem_type == 'Number':
        value = node['number_values'][i]
    elif sem_type == 'Text':
        value = idx_to_text[node['text_values'][i]]
    elif sem_type == 'DateTime':
        value = node['datetime_values'][i]
    elif sem_type == 'Boolean':
        value = node['boolean_values'][i]

    print(f"  {col_name}: {value}")

Table: circuits
  circuitRef of circuits: villeneuve
  name of circuits: Circuit Gilles Villeneuve
  location of circuits: Montreal
  country of circuits: Canada
  lat of circuits: 0.5286135
  lng of circuits: -1.1386287
  alt of circuits: -0.6442801


In [ ]:
from collections import Counter

node_slice = data['node_idxs'][0:10]
counts = Counter(node_slice)

for item, count in counts.items():
    print(f"Node ID: {item}, Count: {count}")

Node ID: 98308, Count: 2
Node ID: 25649, Count: 6
Node ID: 53685, Count: 2


# View Parquet MetaData

In [ ]:
import pyarrow.parquet as pq
import json
import sys
from pathlib import Path

def read_relational_metadata(parquet_path):
    """
    Extract primary key, foreign key, and time column metadata from a parquet file.

    Args:
        parquet_path: Path to the parquet file

    Returns:
        dict with keys: pkey_col, fkey_col_to_pkey_table, time_col
    """
    # Read parquet file metadata
    parquet_file = pq.ParquetFile(parquet_path)
    metadata = parquet_file.schema_arrow.metadata

    if metadata is None:
        print("No metadata found in parquet file")
        return None

    # Decode metadata (it's stored as bytes)
    metadata_dict = {k.decode('utf-8'): v.decode('utf-8') for k, v in metadata.items()}

    # Extract and parse the relational metadata
    result = {}

    # Primary key column
    if 'pkey_col' in metadata_dict:
        result['pkey_col'] = json.loads(metadata_dict['pkey_col'])
    else:
        result['pkey_col'] = None

    # Foreign key columns to primary key tables mapping
    if 'fkey_col_to_pkey_table' in metadata_dict:
        result['fkey_col_to_pkey_table'] = json.loads(metadata_dict['fkey_col_to_pkey_table'])
    else:
        result['fkey_col_to_pkey_table'] = {}

    # Time column
    if 'time_col' in metadata_dict:
        result['time_col'] = json.loads(metadata_dict['time_col'])
    else:
        result['time_col'] = None

    return result


def read_folder_metadata(folder_path, recursive=False):
    """
    Read relational metadata from all parquet files in a folder.

    Args:
        folder_path: Path to the folder containing parquet files
        recursive: If True, search subdirectories as well

    Returns:
        dict mapping file paths to their metadata
    """
    folder = Path(folder_path)

    # Find all parquet files
    if recursive:
        parquet_files = folder.rglob("*.parquet")
    else:
        parquet_files = folder.glob("*.parquet")

    results = {}

    for parquet_file in parquet_files:
        print(f"Processing: {parquet_file.name}")
        try:
            metadata = read_relational_metadata(str(parquet_file))
            results[str(parquet_file)] = metadata
        except Exception as e:
            print(f"  Error reading {parquet_file.name}: {e}")
            results[str(parquet_file)] = None

    return results


read_folder_metadata(os.path.join(os.environ['HOME'], 'relbench_datasets/rel-f1/db/'))

Processing: races.parquet
Processing: drivers.parquet
Processing: constructor_standings.parquet
Processing: standings.parquet
Processing: constructors.parquet
Processing: constructor_results.parquet
Processing: circuits.parquet
Processing: qualifying.parquet
Processing: results.parquet


{'/content/drive/MyDrive/Colab_Data/relbench_datasets/rel-f1/db/races.parquet': {'pkey_col': 'raceId',
  'fkey_col_to_pkey_table': {'circuitId': 'circuits'},
  'time_col': 'date'},
 '/content/drive/MyDrive/Colab_Data/relbench_datasets/rel-f1/db/drivers.parquet': {'pkey_col': 'driverId',
  'fkey_col_to_pkey_table': {},
  'time_col': None},
 '/content/drive/MyDrive/Colab_Data/relbench_datasets/rel-f1/db/constructor_standings.parquet': {'pkey_col': 'constructorStandingsId',
  'fkey_col_to_pkey_table': {'raceId': 'races',
   'constructorId': 'constructors'},
  'time_col': 'date'},
 '/content/drive/MyDrive/Colab_Data/relbench_datasets/rel-f1/db/standings.parquet': {'pkey_col': 'driverStandingsId',
  'fkey_col_to_pkey_table': {'raceId': 'races', 'driverId': 'drivers'},
  'time_col': 'date'},
 '/content/drive/MyDrive/Colab_Data/relbench_datasets/rel-f1/db/constructors.parquet': {'pkey_col': 'constructorId',
  'fkey_col_to_pkey_table': {},
  'time_col': None},
 '/content/drive/MyDrive/Colab_Da

# Load Parquet Data

In [ ]:
import pandas as pd

def load_parquet(file_path):

  # Load the entire file
  df = pd.read_parquet(file_path)

  return df

# Display result
df = load_parquet(os.path.join(os.environ['HOME'], 'relbench_datasets/rel-f1/tasks/driver-dnf/train.parquet'))
print(df.head())

df = load_parquet(os.path.join(os.environ['HOME'], 'relbench_datasets/rel-f1/tasks/driver-position/train.parquet'))
print(df.head())

        date  driverId  did_not_finish
0 1950-05-20       619               1
1 1950-05-20       772               1
2 1950-05-20       780               1
3 1950-06-19       786               0
4 1950-06-19       687               1
        date  driverId  position
0 2004-07-05        10     10.75
1 2004-07-05        47     12.00
2 2004-03-07         7     15.00
3 2004-01-07        10      9.00
4 2003-09-09        52     13.00


# Model Architecture

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from einops import rearrange
from einops._torch_specific import allow_ops_in_compiled_graph
from ml_dtypes import bfloat16
from torch import nn
from torch.nn.attention import SDPBackend, sdpa_kernel
from torch.nn.attention.flex_attention import create_block_mask, flex_attention

In [ ]:
class MaskedAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads

        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, block_mask):
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)

        q = rearrange(q, "b s (h d) -> b h s d", h=self.num_heads)
        k = rearrange(k, "b s (h d) -> b h s d", h=self.num_heads)
        v = rearrange(v, "b s (h d) -> b h s d", h=self.num_heads)

        if block_mask is None:
            with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
                x = F.scaled_dot_product_attention(q, k, v)
        else:
            x = flex_attention(q, k, v, block_mask=block_mask)

        x = rearrange(x, "b h s d -> b s (h d)")
        x = self.wo(x)
        return x

class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class RelationalBlock(nn.Module):
    def __init__(
        self,d_model,num_heads,d_ff,
    ):
        super().__init__()

        self.norms = nn.ModuleDict(
            {l: nn.RMSNorm(d_model) for l in ["feat", "nbr", "col", "full", "ffn"]}
        )
        self.attns = nn.ModuleDict(
            {
                l: MaskedAttention(d_model, num_heads)
                for l in ["feat", "nbr", "col", "full"]
            }
        )
        self.ffn = FFN(d_model, d_ff)

    def forward(self, x, block_masks):
        for l in ["col", "feat", "nbr", "full"]:
            x = x + self.attns[l](self.norms[l](x), block_mask=block_masks[l])
        x = x + self.ffn(self.norms["ffn"](x))
        return x

def _make_block_mask(mask, batch_size, seq_len, device):
  def _mod(b, h, q_idx, kv_idx):
      return mask[b, q_idx, kv_idx]

  return create_block_mask(
      mask_mod=_mod,
      B=batch_size,
      H=None,
      Q_LEN=seq_len,
      KV_LEN=seq_len,
      device=device,
      _compile=False, # Changed to False
  )

In [ ]:
from functools import partial

class RelationalTransformer(nn.Module):
    def __init__(
        self,
        num_blocks,
        d_model,
        d_text,
        num_heads,
        d_ff,
    ):
        super().__init__()

        self.enc_dict = nn.ModuleDict(
            {
                "number": nn.Linear(1, d_model, bias=True),
                "text": nn.Linear(d_text, d_model, bias=True),
                "datetime": nn.Linear(1, d_model, bias=True),
                "col_name": nn.Linear(d_text, d_model, bias=True),
                "boolean": nn.Linear(1, d_model, bias=True),
            }
        )
        self.dec_dict = nn.ModuleDict(
            {
                "number": nn.Linear(d_model, 1, bias=True),
                "text": nn.Linear(d_model, d_text, bias=True),
                "datetime": nn.Linear(d_model, 1, bias=True),
                "boolean": nn.Linear(d_model, 1, bias=True),
            }
        )
        self.norm_dict = nn.ModuleDict(
            {
                "number": nn.RMSNorm(d_model),
                "text": nn.RMSNorm(d_model),
                "datetime": nn.RMSNorm(d_model),
                "col_name": nn.RMSNorm(d_model),
                "boolean": nn.RMSNorm(d_model),
            }
        )
        self.mask_embs = nn.ParameterDict(
            {
                t: nn.Parameter(torch.randn(d_model))
                for t in ["number", "text", "datetime", "boolean"]
            }
        )
        self.blocks = nn.ModuleList(
            [RelationalBlock(d_model, num_heads, d_ff) for i in range(num_blocks)]
        )
        self.norm_out = nn.RMSNorm(d_model)
        self.d_model = d_model

    def forward(self, batch):
        node_idxs = batch["node_idxs"]
        f2p_nbr_idxs = batch["f2p_nbr_idxs"]
        col_name_idxs = batch["col_name_idxs"]
        table_name_idxs = batch["table_name_idxs"]
        is_padding = batch["is_padding"]
        batch_size, seq_len = node_idxs.shape

        batch_size, seq_len = node_idxs.shape
        device = node_idxs.device

        # Padding mask for attention pairs (allow only non-pad -> non-pad)
        pad = (~is_padding[:, :, None]) & (~is_padding[:, None, :])  # (B, S, S)

        # cells in the same node
        same_node = node_idxs[:, :, None] == node_idxs[:, None, :]  # (B, S, S)

        # kv index is among q's foreign -> primary neighbors
        kv_in_f2p = (node_idxs[:, None, :, None] == f2p_nbr_idxs[:, :, None, :]).any(
            -1
        )  # (B, S, S)

        # q index is among kv's primary -> foreign neighbors (reverse relation)
        q_in_f2p = (node_idxs[:, :, None, None] == f2p_nbr_idxs[:, None, :, :]).any(
            -1
        )  # (B, S, S)

        # Same column AND same table
        same_col_table = (col_name_idxs[:, :, None] == col_name_idxs[:, None, :]) & (
            table_name_idxs[:, :, None] == table_name_idxs[:, None, :]
        )  # (B, S, S)

        # Final boolean masks (apply padding once here)
        attn_masks = {
            "feat": (same_node | kv_in_f2p) & pad,
            "nbr": q_in_f2p & pad,
            "col": same_col_table & pad,
            "full": pad,
        }

        # Make them contiguous for better kernel performance
        for l in attn_masks:
            attn_masks[l] = attn_masks[l].contiguous()

        # Convert to block masks
        make_block_mask = partial(
            _make_block_mask,
            batch_size=batch_size,
            seq_len=seq_len,
            device=device,
        )
        block_masks = {
            l: make_block_mask(attn_mask) for l, attn_mask in attn_masks.items()
        }

        x = 0
        x = x + (
            self.norm_dict["col_name"](
                self.enc_dict["col_name"](batch["col_name_values"])
            )
            * (~is_padding)[..., None]
        )

        for i, t in enumerate(["number", "text", "datetime", "boolean"]):
            x = x + (
                self.norm_dict[t](self.enc_dict[t](batch[t + "_values"]))
                * ((batch["sem_types"] == i) & ~batch["masks"] & ~is_padding)[..., None]
            )
            x = x + (
                self.mask_embs[t]
                * ((batch["sem_types"] == i) & batch["masks"] & ~is_padding)[..., None]
            )

        for i, block in enumerate(self.blocks):
            x = block(x, block_masks)

        x = self.norm_out(x)

        loss_out = x.new_zeros(())
        yhat_out = {"number": None, "text": None, "datetime": None, "boolean": None}

        B, S, _ = x.shape
        sem_types = batch["sem_types"]  # (B,S) ints 0..3
        masks = batch["masks"].bool()  # (B,S) where to train

        for i, t in enumerate(["number", "text", "datetime", "boolean"]):
            yhat = self.dec_dict[t](x)  # (B,S, D_t)
            y = batch[f"{t}_values"]  # (B,S, D_y)
            sem_type_mask = (sem_types == i) & masks  # (B,S) mask for this type

            if not sem_type_mask.any():
                if t in yhat_out:
                    # still touch the param to avoid unused param error
                    loss_out = loss_out + (yhat.sum() * 0.0)
                    yhat_out[t] = yhat
                continue

            if t in ("number", "datetime"):
                loss_t = F.huber_loss(yhat, y, reduction="none").mean(-1)
            elif t == "boolean":
                loss_t = F.binary_cross_entropy_with_logits(
                    yhat, (y > 0).float(), reduction="none"
                ).mean(-1)
            elif t == "text":
                raise ValueError("masking text not supported")

            # masked sum for this type
            loss_out = loss_out + (loss_t * sem_type_mask).sum()

            if t in yhat_out:
                yhat_out[t] = yhat

        loss_out = loss_out / masks.sum()

        return loss_out, yhat_out

# Train Model

In [ ]:
%cd relational-transformer

/content/relational-transformer


In [ ]:
def all_gather_nd(tensor: torch.Tensor) -> list[torch.Tensor]:
    """
    Gathers tensor arrays of different lengths in a list.
    The length dimension is 0. This supports any number of extra dimensions in the tensors.
    All the other dimensions should be equal between the tensors.
    Adapted from: https://stackoverflow.com/a/71433508

    Args:
        tensor (Tensor): Tensor to be broadcast from current process.

    Returns:
        list[Tensor]: List of tensors gathered from all processes.
    """
    world_size = dist.get_world_size()
    local_size = torch.tensor(tensor.size(), device=tensor.device)
    all_sizes = [torch.zeros_like(local_size) for _ in range(world_size)]
    dist.all_gather(all_sizes, local_size)

    max_length = max(size[0] for size in all_sizes)

    length_diff = max_length.item() - local_size[0].item()
    if length_diff:
        pad_size = (length_diff, *tensor.size()[1:])
        padding = torch.zeros(pad_size, device=tensor.device, dtype=tensor.dtype)
        tensor = torch.cat((tensor, padding))

    all_tensors_padded = [torch.zeros_like(tensor) for _ in range(world_size)]
    dist.all_gather(all_tensors_padded, tensor)
    all_tensors = []
    for tensor_, size in zip(all_tensors_padded, all_sizes):
        all_tensors.append(tensor_[: size[0]])
    return all_tensors


In [ ]:
from rt.tasks import all_tasks, forecast_tasks
train_tasks=[t for t in all_tasks if t[0] != "rel-amazon"],
eval_tasks=[t for t in forecast_tasks if t[0] == "rel-amazon"],
train_tasks

([('rel-hm', 'user-churn', 'churn', []),
  ('rel-stack', 'user-badge', 'WillGetBadge', []),
  ('rel-stack', 'user-engagement', 'contribution', []),
  ('rel-avito', 'user-visits', 'num_click', []),
  ('rel-avito', 'user-clicks', 'num_click', []),
  ('rel-event', 'user-ignore', 'target', []),
  ('rel-trial', 'study-outcome', 'outcome', []),
  ('rel-f1', 'driver-dnf', 'did_not_finish', []),
  ('rel-event', 'user-repeat', 'target', []),
  ('rel-f1', 'driver-top3', 'qualifying', []),
  ('rel-hm', 'item-sales', 'sales', []),
  ('rel-stack', 'post-votes', 'popularity', []),
  ('rel-trial', 'site-success', 'success_rate', []),
  ('rel-trial', 'study-adverse', 'num_of_adverse_events', []),
  ('rel-event', 'user-attendance', 'target', []),
  ('rel-f1', 'driver-position', 'position', []),
  ('rel-avito', 'ad-ctr', 'num_click', []),
  ('rel-avito', 'SearchInfo', 'IsUserLoggedOn', []),
  ('rel-stack', 'postLinks', 'LinkTypeId', []),
  ('rel-trial', 'studies', 'has_dmc', []),
  ('rel-trial',
   'eli

In [ ]:
import torch
from torch.utils.data import DataLoader
from rt.data import RelationalDataset

batch_size=4
seq_len=256
num_workers=0
rank=0
world_size=1
max_bfs_width=32
embedding_model="all-MiniLM-L12-v2"
d_text=384
save_ckpt_dir="ckpts/leave_rel-amazon"

# Your chosen task from rt/tasks.py
db_name, table_name, target_column, columns_to_drop = ('rel-f1', 'driver-dnf', 'did_not_finish', [])

# RelationalDataset expects: (db, table, target, split, columns_to_drop)
task_for_dataset = (db_name, table_name, target_column, "train", columns_to_drop)

dataset = RelationalDataset(
    tasks=[task_for_dataset],
    batch_size=batch_size,
    seq_len=seq_len,
    rank=rank,
    world_size=world_size,
    max_bfs_width=max_bfs_width,
    embedding_model=embedding_model,
    d_text=d_text,
    seed=0,
)

print({"db" :db_name, "Table": table_name, "target column": target_column, "train_test": "train", "cols_to_drop": columns_to_drop})

{'db': 'rel-f1', 'Table': 'driver-dnf', 'target column': 'did_not_finish', 'train_test': 'train', 'cols_to_drop': []}


In [ ]:
loader = DataLoader(dataset, batch_size=None, num_workers=num_workers
                    , persistent_workers=False
                    ,pin_memory=True
                    ,in_order=True)
batch = next(iter(loader))

print("keys:", batch.keys())

eval_splits=["val", "test"]

eval_loaders = {}
# for db_name, table_name, target_column, columns_to_drop in eval_tasks:
#     for split in eval_splits:
#         eval_dataset = RelationalDataset(
#             tasks=[(db_name, table_name, target_column, split, columns_to_drop)],
#             batch_size=batch_size,
#             seq_len=seq_len,
#             rank=rank,
#             world_size=world_size,
#             max_bfs_width=max_bfs_width,
#             embedding_model=embedding_model,
#             d_text=d_text,
#             seed=0,
#         )
#         eval_dataset.sampler.shuffle_py(0)
#         eval_loaders[(db_name, table_name, split)] = DataLoader(
#             eval_dataset,
#             batch_size=None,
#             num_workers=num_workers,
#             persistent_workers=True,
#             pin_memory=True,
#             in_order=True,
#         )

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


keys: dict_keys(['node_idxs', 'f2p_nbr_idxs', 'table_name_idxs', 'col_name_idxs', 'class_value_idxs', 'col_name_values', 'sem_types', 'number_values', 'text_values', 'datetime_values', 'boolean_values', 'masks', 'is_targets', 'is_task_nodes', 'is_padding', 'true_batch_size'])


In [ ]:
is_padding = batch["is_padding"]
pad = (~is_padding[:, :, None]) & (~is_padding[:, None, :])  # (B, S, S)

In [ ]:
node_idxs = batch["node_idxs"]
print("node_idx shape:", node_idxs.shape)
same_node = node_idxs[:, :, None] == node_idxs[:, None, :]
print("same_node shape:", same_node.shape)
same_node[0][2]

node_idx shape: torch.Size([4, 256])
same_node shape: torch.Size([4, 256, 256])


tensor([False, False,  True,  True,  True,  True,  True,  True, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, 

In [ ]:
node_idxs[0]

tensor([ 98308,  98308,  25649,  25649,  25649,  25649,  25649,  25649, 101176,
        101176,  27426,  27426,  27426,  25435,  25435,  25435,  36888,  36888,
         36888,  36888,  36888,      3,      3,      3,      3,      3,      3,
             3,  53953,  53953,  53953,  53953,  53953,  53953,  53953,  25439,
         25439,  25439,  36941,  36941,  36941,  36941,  36941,     10,     10,
            10,     10,     10,     10,     10,  98378,  98378, 101245, 101245,
        106215, 106215, 104781, 104781,  54163,  54163,  54163,  54163,  54163,
         54163,  54163,  54163,  36950,  36950,  36950,  36950,  36950,      8,
             8,      8,      8,      8,      8,      8, 101189, 101189, 106236,
        106236,  99023,  99023, 107573, 107573, 103321, 103321, 106977, 106977,
        104834, 104834,  98377,  98377, 101877, 101877, 105565, 105565, 105529,
        105529,  55724,  55724,  55724,  55724,  55724,  55724,  55724,  55724,
         55724,  55724,  55724,  25432, 

In [ ]:
batch["table_name_idxs"].shape

torch.Size([4, 256])

In [ ]:
#from rt.model import RelationalTransformer
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import wandb
from torch import optim
from tqdm.auto import tqdm
import time
from sklearn.metrics import r2_score, roc_auc_score
from pathlib import Path
from torch.nn.utils import clip_grads_with_norm_, get_total_norm

os.environ['TORCHDYNAMO_VERBOSE'] = '1'
os.environ['WANDB_NOTEBOOK_NAME'] = 'rt_model_experiment.ipynb'
project="rt"
lr=1e-3 #learning rate
wd=0.1 #weight decay
lr_schedule=True
max_steps=50_001
compile_=False #
max_eval_steps=40
max_grad_norm=1.0

# net = RelationalTransformer(
#     num_blocks=12,
#     d_model=256,
#     d_text=384,
#     num_heads=8,
#     d_ff=1024,
# )

net = RelationalTransformer(
    num_blocks=6,      # Fewer layers for faster iteration
    d_model=128,       # Smaller hidden dimension
    d_text=384,        # Must match embedding dataset
    num_heads=4,       # d_model must be divisible by num_heads
    d_ff=512,          # Usually 2-4x d_model
)


# ddp = "LOCAL_RANK" in os.environ
# device = "cuda"
ddp = False  # Force single-process mode
device = "cpu"  # Use CPU instead of CUDA
if ddp:
    os.environ["OMP_NUM_THREADS"] = f"{num_workers}"
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
    dist.init_process_group("nccl")
if ddp:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
else:
    rank = 0
    world_size = 1

# if rank == 0:
#     run = wandb.init(project=project, config=locals())
#     print(run.name)


if rank == 0:
    param_count = sum(p.numel() for p in net.parameters())
    print(f"{param_count=:_}")

net = net.to(device)
net = net.to(torch.bfloat16)
opt = optim.AdamW(
    net.parameters(),
    lr=lr,
    weight_decay=wd,
    betas=(0.9, 0.999),
    eps=1e-8,
    fused=True,
)

if lr_schedule:
    lrs = optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=lr,
        total_steps=max_steps,
        pct_start=0.2,
        anneal_strategy="linear",
    )

if ddp:
    net = DDP(net)


if compile_:
    net = torch.compile(net, dynamic=False)

steps = 0
# if rank == 0:
#     wandb.log({"epochs": 0}, step=steps)

eval_loader_iters = {}
for k, eval_loader in eval_loaders.items():
    eval_loader_iters[k] = iter(eval_loader)

def evaluate(net):
    metrics = {"val": {}, "test": {}}
    net.eval()
    with torch.inference_mode():
        for (
            db_name,
            table_name,
            split,
        ), eval_loader_iter in eval_loader_iters.items():
            if table_name in [
                "item-sales",
                "user-ltv",
                "item-ltv",
                "post-votes",
                "site-success",
                "study-adverse",
                "user-attendance",
                "driver-position",
                "ad-ctr",
            ]:
                task_type = "reg"
            else:
                task_type = "clf"

            preds = []
            labels = []
            losses = []
            eval_load_times = []
            eval_loader = eval_loaders[(db_name, table_name, split)]
            pbar = tqdm(
                total=(
                    min(max_eval_steps, len(eval_loader))
                    if max_eval_steps > -1
                    else len(eval_loader)
                ),
                desc=f"{db_name}/{table_name}/{split}",
                disable=rank != 0,
            )

            batch_idx = 0
            while True:
                tic = time.time()
                try:
                    batch = next(eval_loader_iter)
                    batch_idx += 1
                except StopIteration:
                    break
                toc = time.time()
                pbar.update(1)

                eval_load_time = toc - tic
                if rank == 0:
                    eval_load_times.append(eval_load_time)

                true_batch_size = batch.pop("true_batch_size")
                for k in batch:
                    batch[k] = batch[k].to(device, non_blocking=True)

                batch["masks"][true_batch_size:, :] = False
                batch["is_targets"][true_batch_size:, :] = False
                batch["is_padding"][true_batch_size:, :] = True

                loss, yhat_dict = net(batch)

                if task_type == "clf":
                    yhat = yhat_dict["boolean"][batch["is_targets"]]
                    y = batch["boolean_values"][batch["is_targets"]].flatten()
                elif task_type == "reg":
                    yhat = yhat_dict["number"][batch["is_targets"]]
                    y = batch["number_values"][batch["is_targets"]].flatten()

                assert yhat.size(0) == true_batch_size
                assert y.size(0) == true_batch_size

                pred = yhat.flatten()

                losses.append(loss.item())
                preds.append(pred)
                labels.append(y)

                if max_eval_steps > -1 and batch_idx >= max_eval_steps:
                    break

            eval_loader_iters[(db_name, table_name, split)] = iter(eval_loader)

            pbar.close()
            preds = torch.cat(preds, dim=0)
            labels = torch.cat(labels, dim=0)

            if ddp:
                # ensure the predictions and labels are gathered jointly
                preds = all_gather_nd(preds)
                labels = all_gather_nd(labels)
            else:
                preds = [preds]
                labels = [labels]

            if rank == 0:
                loss = sum(losses) / len(losses)
                k = f"loss/{db_name}/{table_name}/{split}"
                avg_eval_load_time = sum(eval_load_times) / len(eval_load_times)
                wandb.log(
                    {
                        k: loss,
                        f"avg_eval_load_time/{db_name}/{table_name}": avg_eval_load_time,
                    },
                    step=steps,
                )

                preds = torch.cat(preds, dim=0).float().cpu().numpy()
                labels = torch.cat(labels, dim=0).float().cpu().numpy()

                if task_type == "reg":
                    metric_name = "r2"
                    metric = r2_score(labels, preds)
                elif task_type == "clf":
                    metric_name = "auc"
                    labels = [int(x > 0) for x in labels]
                    metric = roc_auc_score(labels, preds)

                k = f"{metric_name}/{db_name}/{table_name}/{split}"
                wandb.log({k: metric}, step=steps)
                print(f"\nstep={steps}, \t{k}: {metric}")
                metrics[split][(db_name, table_name)] = metric

    return metrics

def checkpoint(best=False, db_name="", table_name=""):
    if rank != 0:
        return
    save_ckpt_dir_ = Path(save_ckpt_dir).expanduser()
    save_ckpt_dir_.mkdir(parents=True, exist_ok=True)
    if best:
        save_ckpt_path = f"{save_ckpt_dir_}/{db_name}_{table_name}_best.pt"
    else:
        save_ckpt_path = f"{save_ckpt_dir_}/{steps=}.pt"

    state_dict = net.module.state_dict() if ddp else net.state_dict()
    torch.save(state_dict, save_ckpt_path)
    print(f"saved checkpoint to {save_ckpt_path}")

pbar = tqdm(
    total=max_steps,
    desc="steps",
    disable=rank != 0,
)

best_val_metrics = dict()
best_test_metrics = dict()

max_steps = 5
while steps < max_steps:
  loader.dataset.sampler.shuffle_py(int(steps / len(loader)))
  loader_iter = iter(loader)


  net.train()

  tic = time.time()

  try:
    batch = next(loader_iter)
  except StopIteration:
    break

  batch.pop("true_batch_size")

  for k in batch: #for every k i.e. node_idxs, table_name_idxs etc.
    batch[k] = batch[k].to(device, non_blocking=True)


  toc = time.time()
  load_time = toc - tic

  # if rank == 0:
  #   wandb.log({"load_time": load_time}, step=steps)

  loss, _yhat_dict = net(batch)
  opt.zero_grad(set_to_none=True)
  loss.backward()


  grad_norm = get_total_norm(
    [p.grad for p in net.parameters() if p.grad is not None]
  )
  clip_grads_with_norm_(
    net.parameters(), max_norm=max_grad_norm, total_norm=grad_norm
  )

  opt.step()

  if lr_schedule:
    lrs.step()

  steps += 1
  print(f"Step: {steps} -  Loss: {loss}")




param_count=2_906_883


steps:   0%|          | 0/50001 [00:00<?, ?it/s]

node_idxs
f2p_nbr_idxs
table_name_idxs
col_name_idxs
class_value_idxs
col_name_values
sem_types
number_values
text_values
datetime_values
boolean_values
masks
is_targets
is_task_nodes
is_padding


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step: 1 -  Loss: 0.91796875


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step: 3 -  Loss: 0.7890625
node_idxs
f2p_nbr_idxs
table_name_idxs
col_name_idxs
class_value_idxs
col_name_values
sem_types
number_values
text_values
datetime_values
boolean_values
masks
is_targets
is_task_nodes
is_padding


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step: 4 -  Loss: 0.70947265625
node_idxs
f2p_nbr_idxs
table_name_idxs
col_name_idxs
class_value_idxs
col_name_values
sem_types
number_values
text_values
datetime_values
boolean_values
masks
is_targets
is_task_nodes
is_padding


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
loss

tensor(0.6953, grad_fn=<DivBackward0>)